# Phase 1 – Data Cleaning & Quality Check

## Objective

The objective of this phase is to examine the quality of the IndiaKart dataset before analysis. This includes identifying missing values, duplicate records, incorrect data types, invalid values, outliers, and referential integrity issues to ensure accurate and reliable analysis.

In [2]:
import pandas as pd

file = r"C:\Users\Mayuri\Desktop\IndiaKart\All_CSV_Converted_Into_Sheets.xlsx"

customers   = pd.read_excel(file, sheet_name='customers DA')
orders      = pd.read_excel(file, sheet_name='orders DA')
order_items = pd.read_excel(file, sheet_name='order_items DA')
payments    = pd.read_excel(file, sheet_name='payments DA')
products    = pd.read_excel(file, sheet_name='products DA')
inventory   = pd.read_excel(file, sheet_name='inventory DA')
returns     = pd.read_excel(file, sheet_name='returns DA')
suppliers   = pd.read_excel(file, sheet_name='suppliers DA')

In [3]:
print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("payments:", payments.shape)
print("products:", products.shape)
print("inventory:", inventory.shape)
print("returns:", returns.shape)
print("suppliers:", suppliers.shape)

customers: (10000, 17)
orders: (50000, 19)
order_items: (100000, 11)
payments: (50000, 13)
products: (1000, 17)
inventory: (1000, 11)
returns: (10000, 10)
suppliers: (200, 14)


In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
orders['delivered_date'] = pd.to_datetime(orders['delivered_date'], errors='coerce')

C:\Users\Mayuri\AppData\Local\Temp\ipykernel_18848\4286724174.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
C:\Users\Mayuri\AppData\Local\Temp\ipykernel_18848\4286724174.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  orders['delivered_date'] = pd.to_datetime(orders['delivered_date'], errors='coerce')


In [5]:
bad_dates = orders[orders['delivered_date'] < orders['order_date']]
print("Orders with delivered_date before order_date:", len(bad_dates))

Orders with delivered_date before order_date: 0


In [6]:
print(payments['status'].value_counts())

failure_rate = (payments['status'] == 'Failed').mean() * 100
print(f"Failure rate: {failure_rate:.2f}%")

status
Success    48252
Failed      1748
Name: count, dtype: int64
Failure rate: 3.50%


In [7]:
orphan_returns = returns[~returns['order_id'].isin(orders['order_id'])]
print("Returns with no matching order_id:", len(orphan_returns))

Returns with no matching order_id: 0


In [8]:
print(inventory['status'].value_counts())

status
In Stock        922
Low Stock        76
Out of Stock      2
Name: count, dtype: int64


In [9]:
zero_order_custs = customers[customers['total_orders'] == 0]
print("Customers with 0 orders:", len(zero_order_custs))

Customers with 0 orders: 457


In [10]:
tables = {
    'customers': customers, 'orders': orders, 'order_items': order_items,
    'payments': payments, 'products': products, 'inventory': inventory,
    'returns': returns, 'suppliers': suppliers
}

for name, df in tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"--- {name} ---")
    print(missing if len(missing) else "No missing values")

--- customers ---
No missing values
--- orders ---
delivered_date    17501
dtype: int64
--- order_items ---
No missing values
--- payments ---
refund_date    50000
dtype: int64
--- products ---
No missing values
--- inventory ---
No missing values
--- returns ---
No missing values
--- suppliers ---
No missing values


In [11]:
print("Duplicate order_id:", orders['order_id'].duplicated().sum())
print("Duplicate customer_id:", customers['customer_id'].duplicated().sum())
print("Duplicate product_id:", products['product_id'].duplicated().sum())
print("Duplicate item_id:", order_items['item_id'].duplicated().sum())

Duplicate order_id: 0
Duplicate customer_id: 0
Duplicate product_id: 0
Duplicate item_id: 0


Payment failure rate is 3.5%. No orphan returns. 457 customers have never ordered (expected, not an error). No unexpected missing values.